# Batch Galaxy Stellar Mass Map Calculation with HEALPix

This notebook contains the complete, reproducible pipeline for calculating and exporting integrated galaxy stellar mass maps. It filters a local universe galaxy catalog across multiple distance thresholds ($30 \text{ to } 200 \text{ Mpc}$) and projects the data onto a spherical HEALPix grid over various angular resolutions ($\text{NSIDE} = 1 \text{ to } 256$).

## Computational Environment & Dependencies

To ensure absolute scientific reproducibility, this notebook requires specific astronomical coordinate transformation libraries alongside standard data science packages. The exact versions used during development are pinned in the root directory's `requirements.txt`.

### Core Astronomical & Numerical Packages:
* **Python Version:** `3.10` or higher (recommended)
* **healpy (v1.16.x or higher):** Python wrapper for HEALPix (Hierarchical Equal Area isoLatitude Pixelation). Used for fast spherical pixel querying (`ang2pix`), pixel coordinate resolution (`pix2ang`), and map generation (`nside2npix`).
* **pandas & numpy:** Used for high-efficiency memory handling of the full galaxy catalog and fast array-based vectorization (`np.bincount`) during mass summation.
* **matplotlib:** (Optional dependency via healpy) Used internally for coordinate projections and visual validation.

### Core Computational Logic Note:
To avoid physical bias during pixel accumulation, the script converts logarithmic stellar masses ($\log M_{\star}$) into linear space ($10^{\log M_{\star}}$), performs a fast histogram summation via `np.bincount` inside each equal-area HEALPix pixel, and seamlessly transforms the total integrated pixel mass back into standard logarithmic form ($\log M_{\odot}$).

### Environment Reproduction:
You can quickly set up your local or server environment to match this notebook using:
```bash
pip install -r requirements.txt

In [ ]:
"""
Batch Galaxy Stellar Mass Map Calculation with HEALPix
Workflow:
    1. Load galaxy catalog (read once to reduce I/O overhead)
    2. Filter galaxies by distance threshold and valid stellar mass
    3. Iterate over multiple HEALPix NSIDE resolutions
    4. Calculate total stellar mass for each HEALPix pixel
    5. Convert total mass back to logarithmic form
    6. Export results to TXT and NPZ files with auto directory creation

HEALPix Angular Resolution Lookup:
    NSIDE  ->  Pixel Angular Size
"""
import os
import warnings
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from healpy.newvisufunc import projview

# -------------------------- Global Configuration & Constants --------------------------
# Suppress non-critical runtime warnings
warnings.filterwarnings("ignore")

# Mapping: HEALPix NSIDE -> Angular resolution label (for directory & filename)
NSIDE_RES_MAPPING = {
    1: '58.6deg',
    2: '29.3deg',
    4: '14.7deg',
    8: '7.33deg',
    16: '3.66deg',
    32: '1.83deg',
    64: '55arcmin',
    128: '27.5arcmin'
    # 256: '13.7arcmin',
    # 512: '6.87arcmin',
    # 1024: '3.44arcmin',
    # 2048: '1.72arcmin'
}

# Select target HEALPix NSIDE list for batch processing
TARGET_NSIDES = [1, 2, 4, 8, 16, 32, 64, 128]

# Distance threshold range: 30 ~ 200 Mpc, step = 10 Mpc
DISTANCE_RANGE = range(30, 210, 10)

# Fixed computational constants
EMPTY_PIXEL_FILL = 1e-10    # Default value for pixels without galaxies
SAVE_DECIMAL_PLACES = 2     # Decimal precision for final log10(mass)

# File paths (modify according to your local/server environment)
INPUT_CATALOG_PATH = "/Users/cyh/Mass_SFR/newVersion/data/Allgalaxy_final_version.csv"
BASE_OUTPUT_DIR = "/Users/cyh/Mass_SFR/newVersion/data/mapdate/MASS"

# -------------------------- Core Function Definition --------------------------
def calculate_and_export_mass_map(
    nside: int,
    res_label: str,
    distance_cut: float,
    raw_catalog: pd.DataFrame
) -> None:
    """
    Compute total stellar mass per HEALPix pixel and export results to files.

    Parameters
    ----------
    nside : int
        HEALPix NSIDE resolution parameter
    res_label : str
        Angular resolution string, used for directory and file naming
    distance_cut : float
        Maximum distance threshold (Mpc), only galaxies with D <= cut are retained
    raw_catalog : pd.DataFrame
        Full loaded galaxy catalog DataFrame

    Filter Rules
    -----------
    1. Galaxy distance D <= distance_cut
    2. Exclude samples with logM = 0 (invalid mass)

    Output Files
    ------------
    - TXT: pixel ID, RA (deg), Dec (deg), log10(total stellar mass)
    - NPZ: Structured data for fast loading (npix, ra, dec, value)
    Files are saved to: {BASE_OUTPUT_DIR}/{res_label}/
    """
    # Step 1: Filter valid galaxies
    dist_mask = raw_catalog["D"] <= distance_cut
    mass_mask = raw_catalog["logM"] != 0
    total_mask = dist_mask & mass_mask
    filtered_data = raw_catalog[total_mask]

    ra_arr = filtered_data["ra"].values
    dec_arr = filtered_data["dec"].values
    log_mass = filtered_data["logM"].values

    # Step 2: Convert logarithmic mass to linear mass for summation
    mass_linear = 10 ** log_mass

    # Step 3: Initialize HEALPix map
    n_pix = hp.nside2npix(nside)
    mass_map = np.full(n_pix, EMPTY_PIXEL_FILL, dtype=np.float64)

    # Get HEALPix pixel index for each galaxy (lonlat=True: RA, Dec in degrees)
    pixel_idx = hp.ang2pix(nside, ra_arr, dec_arr, lonlat=True)

    # Fast summation of linear mass within each pixel
    mass_sum_per_pix = np.bincount(pixel_idx, weights=mass_linear, minlength=n_pix)

    # Convert total linear mass back to log10 form, keep fixed decimal places
    valid_pix_mask = mass_sum_per_pix > 0
    mass_map[valid_pix_mask] = np.round(
        np.log10(mass_sum_per_pix[valid_pix_mask]),
        decimals=SAVE_DECIMAL_PLACES
    )

    # Step 4: Calculate central coordinates for all HEALPix pixels
    all_pixel_ids = np.arange(n_pix)
    ra_center, dec_center = hp.pix2ang(nside, all_pixel_ids, lonlat=True)

    # Step 5: Create output directory automatically
    output_subdir = os.path.join(BASE_OUTPUT_DIR, res_label)
    os.makedirs(output_subdir, exist_ok=True)

    # Define full file paths
    # txt_filepath = os.path.join(output_subdir, f"MASS_D{distance_cut}_{res_label}.txt")
    npz_filepath = os.path.join(output_subdir, f"MASS_D{distance_cut}_{res_label}.npz")

    # Save TXT file
    # txt_output = np.column_stack((all_pixel_ids, ra_center, dec_center, mass_map))
    # np.savetxt(txt_filepath, txt_output)

    # Save NPZ file (compressed structured data)
    np.savez(
        npz_filepath,
        pix_num=all_pixel_ids,
        ra=ra_center,
        dec=dec_center,
        value=mass_map
    )

    # Print formatted progress log
    print(
        f"Completed | NSIDE={nside:3d} | Res={res_label:12s} | Dist={distance_cut:3d} Mpc "
        f"| Valid Galaxies={len(ra_arr):6d}"
    )

# -------------------------- Main Program Entry --------------------------
def main():
    print("=" * 60)
    print("Start Batch HEALPix Stellar Mass Map Calculation")
    print("=" * 60)

    # Pre-check: Verify input catalog exists
    if not os.path.isfile(INPUT_CATALOG_PATH):
        raise FileNotFoundError(f"Error: Input catalog not found -> {INPUT_CATALOG_PATH}")

    # Load galaxy catalog (read once for maximum I/O efficiency)
    print(f"Loading galaxy catalog: {INPUT_CATALOG_PATH}")
    galaxy_catalog = pd.read_csv(INPUT_CATALOG_PATH, low_memory=False)
    total_raw_galaxies = len(galaxy_catalog)
    print(f"Total galaxies in raw catalog: {total_raw_galaxies}\n")

    # Batch loop over all target HEALPix resolutions
    for nside in TARGET_NSIDES:
        current_res = NSIDE_RES_MAPPING[nside]
        print(f"----- Processing NSIDE = {nside} (Resolution: {current_res}) -----")

        # Loop over all distance thresholds
        for dist in DISTANCE_RANGE:
            calculate_and_export_mass_map(
                nside=nside,
                res_label=current_res,
                distance_cut=dist,
                raw_catalog=galaxy_catalog
            )
        print("")

    # Final completion log
    print("=" * 60)
    print("All batch tasks finished successfully!")
    print(f"All results saved to: {BASE_OUTPUT_DIR}")
    print("=" * 60)

# Standard entry point for standalone execution & module import
if __name__ == "__main__":
    main()

Start Batch HEALPix Stellar Mass Map Calculation
Loading galaxy catalog: /Users/cyh/Mass_SFR/newVersion/data/Allgalaxy_final_version.csv
Total galaxies in raw catalog: 364148

----- Processing NSIDE = 1 (Resolution: 58.6deg) -----
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 30 Mpc | Valid Galaxies=  7789
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 40 Mpc | Valid Galaxies= 11424
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 50 Mpc | Valid Galaxies= 15798
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 60 Mpc | Valid Galaxies= 21791
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 70 Mpc | Valid Galaxies= 29983
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 80 Mpc | Valid Galaxies= 39687
Completed | NSIDE=  1 | Res=58.6deg      | Dist= 90 Mpc | Valid Galaxies= 50268
Completed | NSIDE=  1 | Res=58.6deg      | Dist=100 Mpc | Valid Galaxies= 64427
Completed | NSIDE=  1 | Res=58.6deg      | Dist=110 Mpc | Valid Galaxies= 81488
Completed | NSIDE=  1 | Res=58.6deg      | Dist=1